# 01. Data Understanding

**Project:** Aura (Dormathon 2026, Track 1: Predict Early, Decide Better)

**Goal of this notebook:** look at the raw shipment file before we change anything, and decide whether it is good enough to train a disruption-prediction model on.

**Who this is for:** a healthcare logistics coordinator needs to know which shipment will run into trouble *before* it arrives. A model can only help them if the data behind it behaves like real shipments.


### What we check
1. What is in the file (size, columns, types)
2. Is anything missing or duplicated
3. What the numbers and categories look like
4. Do the values make physical sense
5. Do the columns relate to each other the way real shipments would
6. Is there an outcome to predict
7. What we found and what happens next

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)
pio.renderers.default = "plotly_mimetype+notebook_connected"

# Works whether the notebook is run from the project root or from notebooks/
ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
RAW_PATH = ROOT / "data" / "raw" / "global_supply_chain_risk_2026.csv"

# Chart colours, kept the same in every chart
BLUE, ORANGE, AQUA, OCHRE = "#2a78d6", "#eb6834", "#1baf7a", "#a07a12"
INK, MUTED, GRID, SURFACE = "#0b0b0b", "#52514e", "#e6e5e1", "#fcfcfb"
MODE_COLORS = {"Air": BLUE, "Road": ORANGE, "Rail": OCHRE, "Sea": AQUA}
TARGET = "Disruption_Occurred"


def style(fig, title, height=420):
    """Give every chart the same plain look."""
    fig.update_layout(
        title=dict(text=title, font=dict(size=16, color=INK), x=0, xanchor="left"),
        height=height,
        paper_bgcolor=SURFACE,
        plot_bgcolor=SURFACE,
        font=dict(family="Segoe UI, Arial, sans-serif", size=12, color=MUTED),
        margin=dict(l=64, r=32, t=72, b=56),
        hoverlabel=dict(bgcolor="white", font_color=INK),
    )
    fig.update_xaxes(gridcolor=GRID, linecolor=GRID, zeroline=False)
    fig.update_yaxes(gridcolor=GRID, linecolor=GRID, zeroline=False)
    return fig


raw = pd.read_csv(RAW_PATH)
print(f"Loaded {len(raw):,} rows and {raw.shape[1]} columns from {RAW_PATH.relative_to(ROOT)}")

Loaded 5,000 rows and 14 columns from data\raw\global_supply_chain_risk_2026.csv


## 1. What is in the file

One row is one shipment. Here are the first five.

In [2]:
raw.head()

,Shipment_ID,Date,Origin_Port,Destination_Port,Transport_Mode,Product_Category,Distance_km,Weight_MT,Fuel_Price_Index,Geopolitical_Risk_Score,Weather_Condition,Carrier_Reliability_Score,Lead_Time_Days,Disruption_Occurred
0,SC-10000,2025-10-16,Singapore,Los Angeles,Rail,Textiles,5930.83,197.42,2.43,5.0,Hurricane,0.865,41.39,1
1,SC-10001,2024-04-24,Singapore,Shanghai,Rail,Automotive,14285.36,237.24,2.30,7.5,Storm,0.592,40.92,1
2,SC-10002,2024-01-26,Rotterdam,Los Angeles,Rail,Perishables,11113.91,427.42,1.78,5.6,Rain,0.673,11.54,0
3,SC-10003,2024-10-08,Busan,Hamburg,Rail,Electronics,9180.55,170.66,3.20,0.8,Hurricane,0.832,53.13,1
4,SC-10004,2024-09-07,Busan,Singapore,Air,Perishables,2762.27,434.96,2.77,1.9,Fog,0.741,0.50,1


### What each column means

| Column | Meaning | Unit |
|---|---|---|
| `Shipment_ID` | Unique shipment code | text |
| `Date` | Dispatch date | date |
| `Origin_Port` | Port the shipment leaves from | text |
| `Destination_Port` | Port it is going to | text |
| `Transport_Mode` | Air, Sea, Road or Rail | category |
| `Product_Category` | What is being shipped | category |
| `Distance_km` | Length of the route | km |
| `Weight_MT` | Weight of the cargo | metric tonnes |
| `Fuel_Price_Index` | Fuel cost level | index |
| `Geopolitical_Risk_Score` | Risk along the route | 0 to 10 |
| `Weather_Condition` | Weather on the route | category |
| `Carrier_Reliability_Score` | How dependable the carrier is | 0 to 1 |
| `Lead_Time_Days` | Door-to-door delivery time | days |
| `Disruption_Occurred` | 1 if the shipment was disrupted, 0 if not | 0 or 1 |

The file covers general cargo, not only medicine. It has no cold-chain columns (no packaging buffer, storage temperature, cargo value or hospital urgency). Those are added later and are described in notebook 02.

## 2. Is anything missing or duplicated

For each column: its type, how many values are missing, and how many different values it has.

In [3]:
overview = pd.DataFrame({
    "type": raw.dtypes.astype(str),
    "missing": raw.isna().sum(),
    "unique_values": raw.nunique(),
    "example": raw.iloc[0].astype(str),
})
print(f"Missing values in total : {int(raw.isna().sum().sum())}")
print(f"Fully duplicated rows   : {int(raw.duplicated().sum())}")
print(f"Shipment_ID is unique   : {raw['Shipment_ID'].is_unique}")
print(f"Dates run from {raw['Date'].min()} to {raw['Date'].max()}")
overview

Missing values in total : 0
Fully duplicated rows   : 0
Shipment_ID is unique   : True
Dates run from 2024-01-01 to 2025-12-31


,type,missing,unique_values,example
Shipment_ID,str,0,5000,SC-10000
Date,str,0,731,2025-10-16
Origin_Port,str,0,8,Singapore
Destination_Port,str,0,9,Los Angeles
Transport_Mode,str,0,4,Rail
Product_Category,str,0,5,Textiles
Distance_km,float64,0,4998,5930.83
Weight_MT,float64,0,4785,197.42
Fuel_Price_Index,float64,0,331,2.43
Geopolitical_Risk_Score,float64,0,101,5.0


**What this tells us:** the file is complete. No blanks, no duplicate rows, every shipment has its own ID.

A perfectly complete file is itself a sign that the data is generated, since real logistics records always have gaps. The app must still cope with missing input, so we handle that in the code rather than rely on the data being tidy.

## 3. What the numbers and categories look like

### Numeric columns

In [4]:
numeric_cols = [c for c in raw.select_dtypes("number").columns if c != TARGET]
raw[numeric_cols].describe().T.round(2)

,count,mean,std,min,25%,50%,75%,max
Distance_km,5000.0,7704.06,4199.69,500.17,4036.01,7750.12,11347.46,14995.91
Weight_MT,5000.0,246.25,142.52,1.03,124.33,243.50,366.96,499.75
Fuel_Price_Index,5000.0,2.85,0.96,1.20,2.02,2.84,3.71,4.50
Geopolitical_Risk_Score,5000.0,5.08,2.88,0.00,2.60,5.10,7.50,10.00
Carrier_Reliability_Score,5000.0,0.75,0.14,0.50,0.63,0.76,0.88,1.00
Lead_Time_Days,5000.0,19.36,31.41,0.50,2.11,8.25,21.21,236.39


In [5]:
fig = make_subplots(rows=2, cols=3, subplot_titles=numeric_cols,
                    vertical_spacing=0.18, horizontal_spacing=0.07)
for i, col in enumerate(numeric_cols):
    counts, edges = np.histogram(raw[col], bins=40)  # 40 equal-width bins
    centers = (edges[:-1] + edges[1:]) / 2
    fig.add_trace(
        go.Bar(x=centers, y=counts, width=np.diff(edges) * 0.9, marker_color=BLUE,
               hovertemplate=col + ": %{x:,.2f}<br>Shipments: %{y}<extra></extra>"),
        row=i // 3 + 1, col=i % 3 + 1,
    )
fig.update_annotations(font=dict(size=12, color=INK))
style(fig, "How each numeric column is spread (count of shipments)", height=560)
fig.update_layout(showlegend=False)
fig.show()

**What this tells us:**

- `Distance_km`, `Weight_MT`, `Fuel_Price_Index`, `Geopolitical_Risk_Score` and `Carrier_Reliability_Score` are almost flat. Every value is about equally likely, which is how a random number generator behaves and not how real routes behave.
- `Lead_Time_Days` is the only column with a shape. Most shipments take under three weeks, and a long tail runs out to 236 days. Section 4 looks at why.
- `Lead_Time_Days` also has a spike at exactly 0.5 days, the smallest value in the file. That looks like a floor: anything shorter was recorded as half a day.

### Category columns

In [6]:
category_cols = ["Transport_Mode", "Product_Category", "Weather_Condition"]

fig = make_subplots(rows=3, cols=1, subplot_titles=category_cols, vertical_spacing=0.14)
for i, col in enumerate(category_cols, start=1):
    counts = raw[col].value_counts().sort_values()
    share = counts / len(raw)
    fig.add_trace(
        go.Bar(x=counts.values, y=counts.index, orientation="h", marker_color=BLUE,
               text=[f"{c:,}  ({s:.0%})" for c, s in zip(counts, share)],
               textposition="outside", textfont=dict(color=INK), cliponaxis=False,
               hovertemplate="%{y}<br>Shipments: %{x:,}<extra></extra>"),
        row=i, col=1,
    )
fig.update_annotations(font=dict(size=12, color=INK), x=0, xanchor="left")
fig.update_xaxes(range=[0, raw[category_cols].apply(lambda s: s.value_counts().max()).max() * 1.25])
style(fig, "How many shipments fall in each category", height=700)
fig.update_layout(showlegend=False, bargap=0.35, margin=dict(l=140))
fig.show()

In [7]:
ports = pd.DataFrame({
    "shipments_from": raw["Origin_Port"].value_counts(),
    "shipments_to": raw["Destination_Port"].value_counts(),
}).fillna(0).astype(int).sort_values("shipments_to", ascending=False)
print(f"Origin ports: {raw['Origin_Port'].nunique()}, destination ports: {raw['Destination_Port'].nunique()}")
print(f"Shipments where origin and destination are the same port: {(raw['Origin_Port'] == raw['Destination_Port']).sum()}")
ports

Origin ports: 8, destination ports: 9
Shipments where origin and destination are the same port: 0


,shipments_from,shipments_to
Marseille,0,583
Rotterdam,646,574
Busan,667,570
Shanghai,644,561
Dubai,610,559
Antwerp,635,552
Los Angeles,602,537
Hamburg,597,536
Singapore,599,528


**What this tells us:**

- Every category is close to an equal share: four transport modes at about a quarter each, five product types at about a fifth each, five weather conditions at about a fifth each. Real networks are lopsided. Hurricanes do not happen on one shipment in five.
- **Only one shipment in five is pharmaceuticals.** Another fifth is perishables. The remaining three fifths (electronics, textiles, automotive) are not temperature-sensitive at all.
- Nine ports are in the network. Marseille only receives shipments, it never sends any.

## 4. Do the values make physical sense

A model learns whatever patterns are in the data. If the data breaks the laws of physics, the model learns nonsense.

We run four checks.

### Check A: does the same route always have the same distance?

The distance between two ports does not change from one shipment to the next.

In [8]:
routes = (
    raw.groupby(["Origin_Port", "Destination_Port"])["Distance_km"]
    .agg(shipments="count", shortest_km="min", longest_km="max")
    .round(0)
)
routes["longest_vs_shortest"] = (routes["longest_km"] / routes["shortest_km"]).round(1)
inconsistent_routes = int((routes["longest_vs_shortest"] > 1.5).sum())

print(f"Routes in the file: {len(routes)}")
print(f"Routes where the longest distance is over 1.5x the shortest: {inconsistent_routes}")
routes.sort_values("longest_vs_shortest", ascending=False).head(6)

Routes in the file: 64
Routes where the longest distance is over 1.5x the shortest: 64


,,shipments,shortest_km,longest_km,longest_vs_shortest
Origin_Port,Destination_Port,,,,
Busan,Antwerp,85,500.0,14994.0,30.0
Dubai,Antwerp,76,502.0,14938.0,29.8
Rotterdam,Antwerp,83,500.0,14854.0,29.7
Antwerp,Rotterdam,97,503.0,14865.0,29.6
Los Angeles,Marseille,72,509.0,14892.0,29.3
Busan,Shanghai,91,514.0,14984.0,29.2


**Result: fail.** The same pair of ports shows up with distances from a few hundred km to almost 15,000 km. `Distance_km` was drawn at random and is not tied to the route. The port names tell us nothing reliable about how far a shipment travels.

### Check B: are the travel speeds possible?

Speed is distance divided by door-to-door time. Each mode has a top speed it cannot beat. We use generous limits: 950 km/h for air, 120 for rail, 100 for road and 50 for sea.

In [9]:
SPEED_LIMIT_KMH = {"Air": 950, "Rail": 120, "Road": 100, "Sea": 50}

checks = raw.copy()
checks["speed_kmh"] = checks["Distance_km"] / (checks["Lead_Time_Days"] * 24)
checks["too_fast"] = checks["speed_kmh"] > checks["Transport_Mode"].map(SPEED_LIMIT_KMH)

fig = go.Figure()
for mode, color in MODE_COLORS.items():
    fig.add_trace(go.Box(
        y=checks.loc[checks["Transport_Mode"] == mode, "speed_kmh"],
        name=mode, marker_color=color, line_width=2, boxpoints="outliers",
        marker=dict(size=4, opacity=0.5),
    ))
fig.add_trace(go.Scatter(
    x=list(SPEED_LIMIT_KMH), y=list(SPEED_LIMIT_KMH.values()), mode="markers+text",
    marker=dict(symbol="line-ew", size=46, line=dict(color=INK, width=2)),
    text=[f"limit {v}" for v in SPEED_LIMIT_KMH.values()], textposition="top center",
    textfont=dict(color=INK), hoverinfo="skip",
))
fig.update_yaxes(type="log", range=[0, 3.4], tickvals=[1, 3, 10, 30, 100, 300, 1000],
                 title_text="Implied speed (km/h, log scale)")
style(fig, "Implied door-to-door speed by transport mode", height=480)
fig.update_layout(showlegend=False)
fig.show()

speed_table = checks.groupby("Transport_Mode").agg(
    shipments=("too_fast", "size"), faster_than_limit=("too_fast", "sum"),
    median_kmh=("speed_kmh", "median"), fastest_kmh=("speed_kmh", "max")).round(0)
print(f"Shipments faster than their mode allows: {checks['too_fast'].sum()} of {len(checks):,} ({checks['too_fast'].mean():.1%})")
speed_table

Shipments faster than their mode allows: 48 of 5,000 (1.0%)


,shipments,faster_than_limit,median_kmh,fastest_kmh
Transport_Mode,,,,
Air,1320,37,247.0,1249.0
Rail,1185,1,30.0,122.0
Road,1214,7,37.0,164.0
Sea,1281,3,16.0,52.0


**Result: mostly pass.** Fewer than 1 shipment in 100 is faster than its mode allows. Each mode sits in its own speed range, with air far above the others and sea the slowest. Distance and lead time were generated together, mode by mode.

The few impossible rows are short lead times on long routes. Most of them sit on the 0.5-day floor seen in section 3.

### Check C: can a truck or a train actually make that trip?

Los Angeles is separated from every other port in the file by an ocean. A road or rail shipment to or from Los Angeles is impossible.

In [10]:
by_land = checks["Transport_Mode"].isin(["Road", "Rail"])
touches_la = (checks["Origin_Port"] == "Los Angeles") | (checks["Destination_Port"] == "Los Angeles")
checks["land_across_ocean"] = by_land & touches_la

print(f"Road and rail shipments in total             : {by_land.sum():,}")
print(f"Road or rail to or from Los Angeles (ocean)  : {checks['land_across_ocean'].sum():,} ({checks['land_across_ocean'].sum() / by_land.sum():.0%} of them)")
checks[by_land].groupby("Transport_Mode")["land_across_ocean"].agg(shipments="size", across_ocean="sum")

Road and rail shipments in total             : 2,399
Road or rail to or from Los Angeles (ocean)  : 560 (23% of them)


,shipments,across_ocean
Transport_Mode,,
Rail,1185,270
Road,1214,290


**Result: fail.** About a quarter of road and rail shipments cross an ocean. The transport mode was assigned without looking at the ports.

This check only counts the clearest case. Other pairs, such as Singapore to Hamburg by road, are also doubtful, but they are not strictly impossible, so we leave them alone.

### Check D: does the lead time behave like a travel time?

Two things should be true. Longer routes should take longer. And bad weather should slow a shipment down, whatever the mode.

To compare shipments of different lengths fairly, we use **pace**: hours taken per 1,000 km.

In [11]:
checks["pace_h_per_1000km"] = checks["Lead_Time_Days"] * 24 / (checks["Distance_km"] / 1000)
WEATHER_ORDER = ["Clear", "Rain", "Fog", "Storm", "Hurricane"]

pace = checks.pivot_table(index="Weather_Condition", columns="Transport_Mode",
                          values="pace_h_per_1000km", aggfunc="median").loc[WEATHER_ORDER]

fig = go.Figure()
for mode, color in MODE_COLORS.items():
    fig.add_trace(go.Bar(x=pace.index, y=pace[mode], name=mode, marker_color=color,
                         hovertemplate=mode + "<br>%{x}<br>%{y:.1f} hours per 1,000 km<extra></extra>"))
fig.update_yaxes(type="log", title_text="Median hours per 1,000 km (log scale)")
style(fig, "How slowly shipments move in each kind of weather", height=460)
fig.update_layout(barmode="group", bargap=0.25,
                  legend=dict(orientation="h", y=-0.18, x=0, font=dict(color=INK)), margin=dict(b=90))
fig.show()

at_floor = checks["Lead_Time_Days"] <= 0.5
print(f"Link between distance and lead time (correlation): {checks['Distance_km'].corr(checks['Lead_Time_Days']):.2f}")
print(f"Shipments on the 0.5-day floor : {at_floor.sum():,} ({at_floor.mean():.0%}), of which by air: {(at_floor & (checks['Transport_Mode'] == 'Air')).sum():,}")
print(f"Longest lead time              : {checks['Lead_Time_Days'].max():.0f} days")
print()
print("Pace in each weather, as a multiple of the same mode in clear weather:")
(pace / pace.loc["Clear"]).round(1)

Link between distance and lead time (correlation): 0.34
Shipments on the 0.5-day floor : 409 (8%), of which by air: 385
Longest lead time              : 236 days

Pace in each weather, as a multiple of the same mode in clear weather:


Transport_Mode,Air,Rail,Road,Sea
Weather_Condition,,,,
Clear,1.0,1.0,1.0,1.0
Rain,1.1,1.2,1.2,1.2
Fog,1.3,1.5,1.5,1.5
Storm,2.1,3.0,3.0,3.0
Hurricane,5.4,8.1,8.0,8.1


**Result: pass, with one caveat.**

- Longer routes do take longer, and every mode slows down as the weather gets worse. On road, rail and sea, rain adds about a fifth to the travel time, fog about a half, a storm triples it, and a hurricane makes it about eight times longer. Air slows down less: about twice as long in a storm and five times in a hurricane.
- This explains the long tail in section 3. Every lead time above 150 days is a sea shipment caught in a hurricane.
- **The caveat:** the weather is already built into `Lead_Time_Days`. The team treats this column as a planned estimate that is known before any disruption (this is noted in `src/schema.py`). In this file that estimate clearly allows for the weather, so the column carries the weather signal a second time. We keep that in mind when we build features in notebook 03.
- About 8% of shipments sit on the 0.5-day floor, almost all of them by air. Their true lead time may be shorter than recorded.

### Scorecard for all four checks

In [12]:
n = len(checks)
scorecard = pd.DataFrame(
    [
        ("A", "Route has inconsistent distances", f"{inconsistent_routes} of {len(routes)} routes", inconsistent_routes / len(routes), "fail"),
        ("B", "Faster than the mode allows", f"{checks['too_fast'].sum():,} shipments", checks["too_fast"].mean(), "mostly pass"),
        ("C", "Road or rail across an ocean", f"{checks['land_across_ocean'].sum():,} shipments", checks["land_across_ocean"].sum() / by_land.sum(), "fail"),
        ("D", "Lead time on the 0.5-day floor", f"{at_floor.sum():,} shipments", at_floor.mean(), "caveat"),
        ("D", "Lead time grows with distance and bad weather", "all four modes", 1.0, "pass"),
    ],
    columns=["check", "what_we_looked_for", "how_many", "share_of_group", "verdict"],
)
scorecard["share_of_group"] = scorecard["share_of_group"].map("{:.0%}".format)
scorecard

,check,what_we_looked_for,how_many,share_of_group,verdict
0,A,Route has inconsistent distances,64 of 64 routes,100%,fail
1,B,Faster than the mode allows,48 shipments,1%,mostly pass
2,C,Road or rail across an ocean,560 shipments,23%,fail
3,D,Lead time on the 0.5-day floor,409 shipments,8%,caveat
4,D,Lead time grows with distance and bad weather,all four modes,100%,pass


## 5. Do the columns relate to each other

A correlation of 0 means two columns have nothing to do with each other. A value near +1 or -1 means they move together. The outcome column is included as the last row.

In [13]:
corr = raw[numeric_cols + [TARGET]].corr().round(2)

fig = go.Figure(go.Heatmap(
    z=corr.values, x=corr.columns, y=corr.index, zmin=-1, zmax=1,
    colorscale=[[0, "#2a78d6"], [0.5, "#f0efec"], [1, "#e34948"]],
    text=corr.values, texttemplate="%{text:.2f}", textfont=dict(color=INK, size=11),
    xgap=2, ygap=2, colorbar=dict(title="Correlation", thickness=12),
    hovertemplate="%{y}<br>%{x}<br>Correlation: %{z:.2f}<extra></extra>",
))
fig.update_yaxes(autorange="reversed")
fig.update_xaxes(tickangle=-35)
style(fig, "Correlation between numeric columns", height=560)
fig.update_layout(margin=dict(l=190, b=150))
fig.show()

inputs_only = corr.loc[numeric_cols, numeric_cols]
off_diagonal = inputs_only.where(~np.eye(len(inputs_only), dtype=bool)).abs().stack().dropna()
print(f"Strongest link between any two input columns: {off_diagonal.max():.2f}")
print(f"Input pairs with a link stronger than 0.30   : {int((off_diagonal > 0.30).sum() / 2)} of {len(off_diagonal) // 2}")
print()
print("Link between each input and the outcome:")
corr[TARGET].drop(TARGET).sort_values(key=abs, ascending=False).to_frame("correlation")

Strongest link between any two input columns: 0.34
Input pairs with a link stronger than 0.30   : 1 of 15

Link between each input and the outcome:


,correlation
Lead_Time_Days,0.27
Geopolitical_Risk_Score,0.23
Carrier_Reliability_Score,-0.08
Distance_km,0.01
Weight_MT,0.01
Fuel_Price_Index,0.00


**What this tells us:**

- Among the inputs, the only real link is distance with lead time. Everything else was generated independently.
- Three inputs move with the outcome: `Lead_Time_Days`, `Geopolitical_Risk_Score` and, more weakly, `Carrier_Reliability_Score`. Reliable carriers are disrupted a little less, which is the direction we would expect.
- `Distance_km`, `Weight_MT` and `Fuel_Price_Index` show no link to the outcome at all.
- A correlation cannot see text columns. Weather is text, so it is missing from this chart. The next section shows it matters most of all.

## 6. Is there an outcome to predict

A prediction model needs a column that says what actually happened to each shipment. This file has one: `Disruption_Occurred`.

In [14]:
rate = raw[TARGET].mean()
print(f"Shipments disrupted : {int(raw[TARGET].sum()):,} of {len(raw):,} ({rate:.1%})")
print(f"Shipments not disrupted : {int((1 - raw[TARGET]).sum()):,} ({1 - rate:.1%})")

by_weather = raw.groupby("Weather_Condition")[TARGET].agg(["mean", "size"]).loc[WEATHER_ORDER]
fig = go.Figure(go.Bar(
    x=by_weather.index, y=by_weather["mean"], marker_color=BLUE,
    text=[f"{m:.0%}<br>{n:,} shipments" for m, n in zip(by_weather["mean"], by_weather["size"])],
    textposition="outside", textfont=dict(color=INK), cliponaxis=False,
    hovertemplate="%{x}<br>Disrupted: %{y:.1%}<extra></extra>",
))
fig.add_hline(y=rate, line=dict(color=MUTED, width=1, dash="dash"), layer="below")
fig.add_annotation(xref="paper", x=0, y=rate, text=f"All shipments: {rate:.0%}", showarrow=False,
                   xanchor="left", yanchor="bottom", font=dict(color=INK))
fig.update_yaxes(title_text="Share disrupted", tickformat=".0%", range=[0, 1.22])
style(fig, "Disruption rate by weather", height=440)
fig.update_layout(bargap=0.4)
fig.show()

Shipments disrupted : 3,063 of 5,000 (61.3%)
Shipments not disrupted : 1,937 (38.7%)


**What this tells us:**

- **About 6 shipments in 10 are disrupted.** That is very high. In this file a disruption is the normal case, not the rare one.
- **Weather drives it.** Every hurricane shipment is disrupted, and 4 in 5 storm shipments. In clear weather it is still more than 1 in 3.
- **We do not know what "disrupted" means.** The file gives a yes or no. It does not say whether the shipment was late, damaged or spoiled, or how bad it was. So the model can say *whether* trouble is likely, and the file alone cannot say *what kind*.

## 7. What we found and what happens next

### Findings

| # | Finding | Why it matters |
|---|---|---|
| 1 | 5,000 shipments, 14 columns, no missing values, no duplicates | The file is easy to load, but too tidy to be real |
| 2 | General cargo. Only one shipment in five is pharmaceuticals | Aura's cold-chain story covers a part of this file, not all of it |
| 3 | Distance is not tied to the route | Port names cannot be used to judge how far a shipment goes |
| 4 | Travel speeds are possible for 99% of shipments | Distance and lead time agree with each other |
| 5 | About a quarter of road and rail shipments cross an ocean | These rows are doubtful and need a flag |
| 6 | Lead time already allows for the weather | It carries the weather signal a second time, so it looks stronger than it is |
| 7 | There is an outcome column, and 61% of shipments are disrupted | There is something to predict, and the two classes are not far from balanced |
| 8 | Weather is the strongest driver, then geopolitical risk, then the carrier | These three are where the signal is |
| 9 | No cold-chain columns | Buffer, storage standard, cargo value and urgency must be added |

### Decision

The recorded values are usable, so we do not rewrite any of them.

| Keep exactly as recorded | Flag, do not delete | Add (simulated) |
|---|---|---|
| All 14 columns, including the outcome | Road or rail across an ocean | `Elapsed_Transit_Hours_Simulated` (hours travelled so far) |
| | Lead time on the 0.5-day floor | `Storage_Standard_Simulated`, `Remaining_Buffer_Hours_Simulated` |
| | Speed impossible for the mode | `Cargo_Value_USD_Simulated`, `Hospital_Urgency_Tier_Simulated` |

This is done by `python src/build_dataset.py`. It writes two files:

- `data/processed/supply_chain_risk_clean.csv`: the recorded columns plus the flags.
- `data/processed/aura_shipments.csv`: the same rows plus the simulated cold-chain columns. This is the file we train on.

### Next

Notebook 02 checks the rebuilt file, then explores what separates disrupted shipments from safe ones.

### Limits we state openly

- The file looks generated: no gaps, flat spreads, and equal shares in every category. Results show that the method works, not how it would score on a real carrier's records.
- The outcome is a plain yes or no. The kind of disruption is not recorded.
- The cold-chain columns are simulated by us. They are built without looking at the outcome, so they cannot leak the answer, and for the same reason they cannot help predict it.